# Function calling with Agentic RAG, agentic loop

Before: 
```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A
```

* The pipeline is fixed: search, build prompt, LLM.


After: 
```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

* The LLM searched, saw the results were bad, and decided to try again with a different query. It made that decision on its own. 


In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data

documents = load_faq_data()

In [6]:
from sqlitesearch import TextSearchIndex

# 1. Point to your existing db file with the exact same schema fields
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"  # Adjust the path if the file is in a different folder
)

In [7]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=instructions,
)

In [8]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from **https://ollama.com/download** for your operating system:\n   - **macOS**: download the `.pkg`\n   - **Windows**: download the `.msi`\n   - **Linux**: run\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and start a model with:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model and starts a local chat interface.\n\n3. To check that the local server is running, try:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response like:\n   ```json\n   {"models": [...]}\n   ```\n\n4. If you want to use it from Python, install the client:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```'

### Without tools

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—if enrollment is still open, you can usually join the course.\n\nA couple of things to check:\n- **Enrollment deadline**: some courses close registration after the first class or a set date.\n- **Prerequisites**: make sure you meet any required background.\n- **Capacity/waitlist**: if it’s full, you may need to join a waitlist.\n- **Missed material**: if the course already started, ask whether recordings, slides, or notes are available.\n\nIf you want, I can help you draft a quick message to the instructor or registrar asking to join.'

### Add tools - search

In [10]:
def search(query):
    boost_dict = {'question':3.0, "section": 0.5}
    filter_dict = {'course':'llm-zoomcamp'}

    return sqlite_index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict,
        filter_dict = filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for results matching the given query",
    "parameters": {
        "type": "object",
        "properties": {
            "query" :{
                "type":"string",
                "description": "search query text to look up in the knowledge base"
            }
        },
    "required": ["query"],
    "additionalProperties": False
    }
}

### Sending the question with `search` tool

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered just discovered can I join enrollment eligibility FAQ"}', call_id='call_JDs5M8Bi8HmnWJKMZK0moUkB', name='search', type='function_call', id='fc_02e3ea3ba0f1fd12006a41688af384819c9f42fb19ace33a3a', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)

In [14]:
results = search(**args)
result_json = json.dumps(results, indent=2)

Workflow 

1. make a call to the LLM
2. LLM decides to involve search('params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (2nd call)
5. LLM processes the results
6. LLM gives the answer


In [15]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [16]:
# create history 
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})



In [17]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered just discovered can I join enrollment eligibility FAQ"}', call_id='call_JDs5M8Bi8HmnWJKMZK0moUkB', name='search', type='function_call', id='fc_02e3ea3ba0f1fd12006a41688af384819c9f42fb19ace33a3a', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_JDs5M8Bi8HmnWJKMZK0moUkB',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expe

In [18]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [19]:
print(response.output_text)

Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still open.


###  Token usage and cost

In [20]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(792, 29)

In [21]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


### Agentic Loop

Allow the LLM to make loops, e.g. additional tool calls

In [ ]:
instructions = """
You're a course teaching assistant.

You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyse the results, and then perform more searches if needed. 

Try to expand your search by using new keywords based on the results you get from the search.


The question has to be about the course or its logistics, offtopic questions shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.

If you can't find an answer, return `I don't know.`
""".strip()

In [23]:
# A helper function that wraps JSON arguments into a pytho dict

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

* with `while`: Keep looping with tools until there's a text result

In [36]:
question = "can i join course today"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

iteration = 1 

# 1. Start a loop to handle the back-and-forth conversation turns
while True:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # Append the model's response (tool calls or text) to the history
    messages.extend(response.output)
    
    has_function_calls = False

    # 2. Process what the model sent back
    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)  # Add the tool result to history
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    iteration = iteration +1
    # 3. If the model didn't ask for any more functions, it means 
    # it has provided the final text answer, so we can break the loop.
    if not has_function_calls:
        break

function_call: search {"query":"join course today enrollment start date registration deadline add/drop course join today"}
function_call: search {"query":"course enrollment today can i join today registration open deadline FAQ"}
ASSISTANT:
Yes — you can join the course today. The course materials are available, and you can start learning right away.

A couple of notes:
- You can start whenever you want.
- If you want a certificate, you need to submit your project while submissions are still open.
- If you’re asking about registration, it’s mainly for gauging interest and isn’t required to start learning.

If you want, I can also help you with the exact steps to get started or explain the certificate requirements.


### Wrapping the agentic loop in a function

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [46]:
agent_loop(instructions = instructions, question = "how do i run olama")

iteration #1...
function_call: search {"query":"olama run install start ollama how to run"}
function_call: search {"query":"ollama how to run command line start model"}
function_call: search {"query":"ollama installation run model FAQ"}
iteration #2...
ASSISTANT:
If you mean **Ollama**, the basic way to run it is:

1. **Install Ollama**
   - macOS: download from https://ollama.com/download
   - Windows: download the `.msi` from there
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Run a model**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and start a local chat session.

3. **If you need to start the server manually**
   ```bash
   ollama serve
   ```
   If you’re in a notebook or background shell:
   ```bash
   nohup ollama serve > nohup.out 2>&1 &
   ```

4. **Test that it’s working**
   ```bash
   curl http://localhost:11434
   ```

5. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```

'If you mean **Ollama**, the basic way to run it is:\n\n1. **Install Ollama**\n   - macOS: download from https://ollama.com/download\n   - Windows: download the `.msi` from there\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Run a model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed and start a local chat session.\n\n3. **If you need to start the server manually**\n   ```bash\n   ollama serve\n   ```\n   If you’re in a notebook or background shell:\n   ```bash\n   nohup ollama serve > nohup.out 2>&1 &\n   ```\n\n4. **Test that it’s working**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n5. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also s